# 06 — SCP3078 full integrated MCAO atlas (5.0 GB H5AD)

Purpose: characterize frozen PET/BBB target transcript detection in a **harmonized, integrated** mouse tMCAO atlas.

Source: [Kuwata et al. / Single Cell Portal SCP3078](https://singlecell.broadinstitute.org/single_cell/study/SCP3078/mcaoatlas-allcells).

**This is NOT independent replication if its inputs overlap GSE174574, GSE227651 or the other public stroke datasets.** Treat results as sensitivity and cell-type-context evidence until source sample/animal IDs can be traced. The file contains a *normalized expression* matrix, not necessarily raw UMI counts.

**Data handling:** streamed download of source H5AD, SHA256, backed reading and row-chunk processing. Raw data are NOT committed to GitHub. Source title/file/size recorded.


In [ ]:
!pip -q install "anndata>=0.10" "h5py>=3.10" "scanpy>=1.10,<1.12"
from pathlib import Path
import requests, hashlib, os, json, sys, gc
import pandas as pd, numpy as np, scipy.sparse as sp, anndata as ad

ROOT=Path(os.environ.get("IMACS_ROOT","/content/imacs_stroke_pet_atlas_full" if Path("/content").exists() else "imacs_stroke_pet_atlas_full"))
RAW=ROOT/"raw"; OUT=ROOT/"results"
RAW.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
URL="https://singlecell.broadinstitute.org/single_cell/api/v1/site/studies/SCP3078/download?filename=MCAO_norExpr_metaanalysis_all.h5ad"
LOCAL=RAW/"MCAO_norExpr_metaanalysis_all.h5ad"
CAT=pd.read_csv("https://raw.githubusercontent.com/Touf00/retinal-nvu-computational-analysis/imacs-stroke-pet-screen/IMACS_Stroke_PET/config/candidate_catalog.csv").drop_duplicates("gene")


## Stream source file without loading into memory

In [ ]:
if not LOCAL.exists() or LOCAL.stat().st_size<100_000_000:
    if LOCAL.exists():LOCAL.unlink()
    with requests.get(URL,stream=True,timeout=300) as r:
        r.raise_for_status()
        total=int(r.headers.get("Content-Length",0))
        print("Content-Type:",r.headers.get("Content-Type"),"Bytes reported:",total)
        assert (not total or total>100_000_000), "Unexpectedly short download; server returned metadata/error"
        with LOCAL.open("wb") as f:
            for block in r.iter_content(chunk_size=4*1024*1024):
                if block:f.write(block)
assert LOCAL.stat().st_size>100_000_000, "Incomplete large H5AD download"
h=hashlib.sha256()
with LOCAL.open("rb") as f:
    for buf in iter(lambda:f.read(8*1024*1024),b""):h.update(buf)
print("H5AD",LOCAL.stat().st_size,"SHA256",h.hexdigest())
(OUT/"06_SCP3078_input_manifest.json").write_text(json.dumps({
"study":"SCP3078","source":URL,"file":LOCAL.name,
"bytes":LOCAL.stat().st_size,"sha256":h.hexdigest(),
"independence":"unknown until original samples mapped",
"assay":"normalized integrated scRNAseq",
},indent=2))

## Inspect the actual source-study, animal, time and cell type metadata before analysis

**No guessed study IDs.** Metadata fields and example distinct values are exported to a durable audit. Inference is disabled.


In [ ]:
a=ad.read_h5ad(LOCAL,backed="r")
print("Shape",a.shape,"X backing",type(a.X))
print("obs columns",list(a.obs.columns))
print("var columns",list(a.var.columns))
overview=[]
for col in a.obs.columns:
    x=a.obs[col]
    unique=x.nunique(dropna=False)
    examples=x.astype(str).drop_duplicates().head(12).tolist()
    overview.append({"field":col,"unique_values":int(unique),"examples":" | ".join(examples),
        "dtype":str(x.dtype)})
D=pd.DataFrame(overview)
D.to_csv(OUT/"06_SCP3078_obs_metadata_field_audit.csv",index=False)
display(D)

# Treat parsed group labels as descriptive only even if timepoint/source exists
colnames={s.lower():s for s in a.obs.columns}
def first_field(candidates):
    for cand in candidates:
        if cand.lower() in colnames:return colnames[cand.lower()]
    return None

ctcol=first_field(["cell_type__ontology_label","cell_type","celltype","annotation","major_cell_type"])
timecol=first_field(["timepoint","time_point","time","condition","day","stroke_time"])
studycol=first_field(["source_dataset","source_study","orig.ident","study","dataset","sample_id"])
print("Detected",dict(celltype=ctcol,timepoint=timecol,study=studycol))
assert ctcol is not None, "Atlas lacks interpretable cell-type labels; abort rather than invent annotations"


## Process frozen panel by backed matrix blocks (no loading 5 GB expression matrix into RAM)

Rows are summarized only within metadata-defined groups. No p-values, no DESeq2, no meta-analysis.

In [ ]:
gene_index={}
for gene in CAT.gene:
    where=np.flatnonzero(a.var_names==gene)
    if len(where):gene_index[gene]=int(where[0])
missing_genes=sorted(set(CAT.gene)-set(gene_index))
pd.DataFrame({"unmatched_gene_symbol":missing_genes}).to_csv(OUT/"06_SCP3078_unmatched_symbols.csv",index=False)
print("Present genes:",len(gene_index),"missing:",missing_genes)

idx=np.array(list(gene_index.values()),dtype=int)
gnames=list(gene_index)
meta=pd.DataFrame({"cell_type":a.obs[ctcol].astype(str).values,
 "time":a.obs[timecol].astype(str).values if timecol else "__TIME_NOT_IN_METADATA__",
 "study":a.obs[studycol].astype(str).values if studycol else "__STUDY_NOT_IN_METADATA__"})

aggregated=[]
step=500
for start in range(0,a.n_obs,step):
    stop=min(start+step,a.n_obs)
    chunk=a.X[start:stop,:]
    subset=chunk[:,idx]
    subset=subset.toarray() if sp.issparse(subset) else np.asarray(subset)
    m=meta.iloc[start:stop].reset_index(drop=True).copy()
    for i,gene in enumerate(gnames):
        m["value"]=subset[:,i]
        g=m.groupby(["cell_type","time","study"],sort=False)["value"].agg(
          n="size",sum_value="sum",n_detected=lambda v:(v>0).sum()).reset_index()
        g["gene"]=gene
        aggregated.append(g)
    del chunk,subset,m
    if (start//step)%20==0:print("Read backed H5AD rows",stop,"/",a.n_obs)
print("Combining groups:",len(aggregated))
res=pd.concat(aggregated,ignore_index=True)
res=res.groupby(["cell_type","time","study","gene"],as_index=False)[["n","sum_value","n_detected"]].sum()
res["mean_stored_normalized_expression"]=res.sum_value/res.n
res["detection_fraction"]=res.n_detected/res.n
res.to_csv(OUT/"06_SCP3078_gene_celltype_time_study_descriptive.csv",index=False)
display(res.head(35))


In [ ]:
import json
(OUT/"06_SCP3078_critical_limitations.json").write_text(json.dumps({
"source_independence_verified":False,
"source_study_metadata_field":studycol,
"timepoint_metadata_field":timecol,
"celltype_metadata_field":ctcol,
"normalized_not_raw":True,
"unit_of_inference":"NO INFERENCE; descriptive atlas visualization only",
"possible_source_overlap":"GSE174574 and/or GSE227651, not adjudicated",
"source_dataset_matrix_size_bytes":LOCAL.stat().st_size
},indent=2))
a.file.close()
print("Saved atlas screening output:",sorted(p.name for p in OUT.iterdir()))

## Stoplight: integrated atlas limitations

- If study/animal identifiers are absent, do not claim independent replication or patient-level group contrasts.
- If matrix reflects normalized data, detection and mean units must not be called raw UMI counts.
- Integration may remove batch effects, amplify selection bias or obscure temporal effects.
- The atlas is complementary to raw animal-level GSE174574, not a duplicate biological replicate.
- No indication of PET ligand availability, vascular luminal accessibility or BBB transport functionality can be drawn from these RNA summaries.
